# 04. 데이터 전처리 - 실무전용 문제 모음

> **실무 전용 노트북** - 이론 설명 없이 TODO 문제만 모았습니다. (관련 이론 노트북: 04_데이터전처리.ipynb)
이미 개념은 이해했다는 전제로, 손으로 더 많이 풀어보며 완전히 몸에 익히는 것이 목표입니다. 정답을 먼저 보지 말고 반드시 스스로 코드를 작성해본 뒤 펼쳐서 비교하세요.

이론 노트북에서는 Titanic으로 연습했다면, 여기서는 **마케팅 전환 데이터·매장 매출 데이터**로 결측치·이상치·인코딩·스케일링을 다시 연습합니다.

## 목차
1. 결측치 탐색 및 처리
2. 이상치 탐색 및 처리
3. 파생변수 생성 & 구간화
4. 범주형 변수 인코딩
5. 피처 스케일링 & 데이터 분할

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

mkt = pd.read_csv('../data/marketing_conversion.csv')
sales = pd.read_csv('../data/retail_sales.csv')
mkt.head()

## 1. 결측치 탐색 및 처리

**문제 1.** `mkt`의 컬럼별 결측치 개수를 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
print(mkt.isnull().sum())  # 컬럼별 결측치 개수를 먼저 파악해야 어떤 방식으로 채울지 정할 수 있음
```

</details>

**문제 2.** `mkt['나이']`의 결측치를 중앙값으로 채우세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
mkt['나이'] = mkt['나이'].fillna(mkt['나이'].median())  # 나이처럼 이상치(극단값)가 있을 수 있는 변수는 평균보다 중앙값이 더 안전
```

</details>

**문제 3.** `sales['광고비_만원']`의 결측치를 평균으로 채우세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
sales['광고비_만원'] = sales['광고비_만원'].fillna(sales['광고비_만원'].mean())  # 분포가 비교적 고르면 평균 대체도 무난함
```

</details>

**문제 4.** `sales`를 다시 불러온 `sales_ffill`에서 `광고비_만원` 결측치를 '바로 앞 행의 값'으로 채우는 `ffill`을 적용하세요.

In [ ]:
# 여기에 코드를 작성하세요

<details>
<summary>✅ 정답 보기</summary>

```python
sales_ffill = pd.read_csv('../data/retail_sales.csv')  # 문제3에서 이미 채운 원본과 별도로 다시 불러옴
sales_ffill['광고비_만원'] = sales_ffill['광고비_만원'].ffill()  # ffill(): 결측치를 바로 위(앞) 행의 값으로 채움(구버전 표기: fillna(method='ffill')). 반대는 bfill()(바로 아래 값)
sales_ffill['광고비_만원'].isnull().sum()
```

</details>

**문제 5.** `mkt`를 다시 불러온 `mkt_sub`에서, `나이` 컬럼에 결측치가 있는 행만 `dropna(subset=)`로 제거하세요.

In [ ]:
# 여기에 코드를 작성하세요

<details>
<summary>✅ 정답 보기</summary>

```python
mkt_sub = pd.read_csv('../data/marketing_conversion.csv')
mkt_sub = mkt_sub.dropna(subset=['나이'])  # subset을 지정하면 '나이' 컬럼이 결측인 행만 제거(다른 컬럼 결측은 무시)
mkt_sub.shape
```

</details>

**문제 6.** 아래 `toy`처럼 한 행이 통째로 결측치인 경우가 있을 때, `dropna(how='all')`로 '모든 값이 결측인 행'만 제거해보세요.

In [ ]:
toy = pd.DataFrame({'a': [1, None, 3], 'b': [2, None, 4]})
# 여기에 코드를 작성하세요

<details>
<summary>✅ 정답 보기</summary>

```python
toy_clean = toy.dropna(how='all')  # how='any'(기본값)는 하나라도 결측이면 삭제, how='all'은 전부 결측일 때만 삭제
toy_clean
```

</details>

**문제 7.** 아래 `raw`처럼 결측치 대신 물음표(`'?'`)로 표시된 값이 섞여 있을 때, `replace()`로 먼저 `NaN`으로 바꾸고 실수형으로 변환한 뒤, `dropna()`로 결측 행을 제거해보세요.

In [ ]:
raw = pd.DataFrame({'점수': ['85', '?', '92', '78']})
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
raw['점수'] = raw['점수'].replace('?', np.nan).astype(float)  # isnull()로는 안 잡히는 '?' 같은 숨은 결측치를 먼저 NaN으로 바꿔야 dropna/fillna를 쓸 수 있음
raw_clean = raw.dropna()
raw_clean
```

</details>

## 2. 이상치 탐색 및 처리

**문제 8.** `sales['매출_만원']`의 boxplot을 그려 이상치가 있는지 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
sns.boxplot(x=sales['매출_만원'])  # 박스 밖으로 벗어난 점들이 이상치 후보
plt.show()
```

</details>

**문제 9.** `sales['매출_만원']`에서 IQR 기준 이상치 개수를 세는 함수를 작성해 실행하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
def count_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr  # IQR의 1.5배를 벗어나면 이상치로 보는 것이 관례적 기준
    return ((s < lower) | (s > upper)).sum()
print(count_outliers(sales['매출_만원']))
```

</details>

**문제 10.** `sales['매출_만원']`의 이상치를 상한값으로 clip 처리하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
q1, q3 = sales['매출_만원'].quantile([0.25, 0.75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
sales['매출_만원'] = sales['매출_만원'].clip(upper=upper)  # clip은 행을 삭제하지 않고 상한을 넘는 값만 상한값으로 눌러줌(데이터 손실 없이 이상치 완화)
```

</details>

## 3. 파생변수 생성 & 구간화

**문제 11.** `mkt['나이']`를 [0,20,40,60,100] 구간으로 나눠 `연령대` 컬럼을 만드세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
mkt['연령대'] = pd.cut(mkt['나이'], bins=[0, 20, 40, 60, 100], labels=['10대이하', '20-30대', '40-50대', '60대이상'])  # 연속형 나이를 구간별 범주로 나눠 해석하기 쉬운 파생변수를 만듦
mkt['연령대'].value_counts()
```

</details>

**문제 12.** `mkt['나이']`를 `pd.qcut()`으로 데이터 개수가 균등하도록 4개 구간으로 나눠 `나이등급` 컬럼을 만드세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
mkt['나이등급'] = pd.qcut(mkt['나이'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])  # cut()은 값의 '범위'를 균등하게 나누고, qcut()은 '데이터 개수(분위수)'가 균등하도록 나눔
mkt['나이등급'].value_counts()
```

</details>

**문제 13.** `KBinsDiscretizer`로 `mkt[['나이']]`를 5개 구간으로 나눠 `나이_kbins` 컬럼에 저장하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.preprocessing import KBinsDiscretizer

kbins = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')  # encode='ordinal': 구간 번호(0~4)로 반환, strategy='quantile': qcut처럼 데이터 개수가 균등하게 나뉨
mkt['나이_kbins'] = kbins.fit_transform(mkt[['나이']])
mkt['나이_kbins'].value_counts().sort_index()
```

</details>

**문제 14.** `mkt['방문횟수']`가 5 이상이면 '단골', 아니면 '일반'인 `고객유형` 컬럼을 apply로 만드세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
mkt['고객유형'] = mkt['방문횟수'].apply(lambda x: '단골' if x >= 5 else '일반')  # apply+lambda로 조건에 따라 새로운 범주형 파생변수를 생성
mkt['고객유형'].value_counts()
```

</details>

## 4. 범주형 변수 인코딩

**문제 15.** `mkt`의 `기기종류`, `유입경로`를 `get_dummies(drop_first=True)`로 인코딩하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
mkt_enc = pd.get_dummies(mkt, columns=['기기종류', '유입경로'], drop_first=True)  # drop_first=True로 다중공선성을 피하면서 여러 범주형 컬럼을 한 번에 인코딩
mkt_enc.head()
```

</details>

**문제 16.** `sales`의 `지역`을 `LabelEncoder`로 숫자화하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
sales['지역_encoded'] = le.fit_transform(sales['지역'])  # LabelEncoder는 문자열을 0,1,2...로 순서 없이 번호만 매김(트리 모델엔 무난하지만 선형모델엔 부적절할 수 있음)
sales[['지역', '지역_encoded']].drop_duplicates()  # 어떤 지역이 어떤 번호로 매핑됐는지 중복 없이 확인
```

</details>

**문제 17.** `mkt['기기종류']`를 `category` dtype으로, `mkt['방문횟수']`를 `int32`로 변환하고 `기기종류` 컬럼의 변환 전/후 메모리 사용량을 비교하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
before = mkt['기기종류'].memory_usage(deep=True)
mkt['기기종류'] = mkt['기기종류'].astype('category')  # 반복되는 문자열(범주)을 내부적으로 정수 코드로 저장해 메모리를 크게 줄임
after = mkt['기기종류'].memory_usage(deep=True)
print(before, '->', after)

mkt['방문횟수'] = mkt['방문횟수'].astype('int32')  # int64 대신 값의 범위에 맞는 더 작은 정수 타입으로 다운캐스팅해 메모리 절약
print(mkt.dtypes[['기기종류', '방문횟수']])
```

</details>

## 5. 피처 스케일링 & 데이터 분할

**문제 17.** `mkt_enc`에서 `구매여부`를 제외한 수치형 컬럼을 `StandardScaler`로 스케일링하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.preprocessing import StandardScaler
num_cols = mkt_enc.select_dtypes(include='number').columns.drop('구매여부')  # 타깃(구매여부)까지 스케일링하면 안 되므로 미리 제외
scaler = StandardScaler()
mkt_scaled = mkt_enc.copy()  # 원본을 보존하기 위해 복사본에 스케일링 결과를 반영
mkt_scaled[num_cols] = scaler.fit_transform(mkt_enc[num_cols])  # StandardScaler는 평균 0, 표준편차 1로 변환
mkt_scaled[num_cols].describe()
```

</details>

**문제 18.** 같은 컬럼들을 이번엔 `MinMaxScaler`로 스케일링해 `mkt_enc_mm`에 저장하세요.

In [ ]:
# 여기에 코드를 작성하세요

<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.preprocessing import MinMaxScaler
num_cols = mkt_enc.drop(columns=['구매여부']).select_dtypes('number').columns
mm = MinMaxScaler()  # StandardScaler는 평균0/표준편차1, MinMaxScaler는 최소값0~최대값1 범위로 압축
mkt_enc_mm = mkt_enc.copy()
mkt_enc_mm[num_cols] = mm.fit_transform(mkt_enc[num_cols])
mkt_enc_mm[num_cols].describe()
```

</details>

**문제 19.** `mkt_enc`를 `train_test_split(test_size=0.2, stratify=구매여부, random_state=42)`로 분할하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.model_selection import train_test_split
X = mkt_enc.drop(columns=['구매여부'])
y = mkt_enc['구매여부']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)  # stratify=y로 구매여부 비율을 train/test에 동일하게 유지
print(X_train.shape)
```

</details>